# 🐍 Part 2: Database Enrichment & Data Wrangling via Python 
## Refactoring and Enriching the `Reviews` Table (`06_update_reviews.ipynb`) 

## 🎯 Context & Objectives 

The goal of this stage is to perform strategic data enrichment on the `reviews` table within the `insight_places` database using Python (`06_update_reviews.ipynb`). By linking guest evaluations directly to completed bookings (`booking_id`), modeling realistic user review behaviors, establishing strict alignment between numerical ratings (`rating`) and textual feedback (`review_text`), and capturing bidirectional host evaluations (`host_review`), I will transform this table into a valuable feedback dimension for sentiment analysis, property quality scoring, and host performance metrics. 

This notebook represents a key module of a sequential ETL execution, preserving strict relational integrity with the `bookings`, `listings`, and `customers` tables. 


## 🛠️ Environment Setup & Dependencies 

To execute this ETL pipeline securely and efficiently, the following Python libraries are required: * **`sqlalchemy` & `pymysql`**: Database abstraction layer and MySQL driver for executing DDL/DML operations. 
* **`python-dotenv`**: Security protocol to parse database credentials from the existing `.env` environment file without exposing sensitive keys. 
* **`pandas`**: Data manipulation, aggregation, and structural validation. 
* **`random` & `datetime`**: Native modules used to generate realistic behavior probabilities, date offsets, and sentiment-aligned text mappings. 


## 📌 Refining the `Reviews` Table 

This module focuses on upgrading review volume, behavioral distribution, sentiment alignment, and temporal consistency: 
1. **Relational Booking Linkage (`booking_id`):** Restricting evaluation generation exclusively to completed bookings (`booking_status = 'Completed'`), ensuring strict operational logic. 
2. **Behavioral Archetype Distribution:** Modeling real-world guest engagement probabilities: Rating & Text (65%), Rating Only (15%), Text Only (5%), and Non-responding guests (15%). 
3. **Sentiment & Rating Alignment:** Synchronizing numerical ratings (1 to 5) with category-specific text reviews (`review_text`) to maintain natural language consistency for BI sentiment analytics. 
4. **Bidirectional Host Feedback (`host_review`):** Capturing host evaluations regarding guest behavior (e.g., policy compliance, property condition upon check-out). 
5. **Post-Checkout Temporal Rule (`created_at`):** Generating review submission timestamps strictly within a realistic 1 to 7 day window following the booking check-out date (`end_date`). 


### 1) Installing and importing the libraries 
In this step, I will load the libraries that will be used throughout the project. 


In [2]:
!pip install sqlalchemy
!pip install pymysql
import pandas as pd
import random
import datetime
from sqlalchemy import create_engine, text
import pymysql

### 1.1) Installing the data protection library
I will install the python-dotenv library to ensure the security of the MySQL password



In [3]:
!pip install python-dotenv

### 1.2) Environment Setup and Security Validation

In [4]:
import os
from dotenv import load_dotenv

# Loading variables from the .env file
load_dotenv()

# Assigning credentials to variables using os.getenv
user = os.getenv('DB_USER')
password = os.getenv('DB_PASSWORD')
host = os.getenv('DB_HOST', 'localhost')
port = os.getenv('DB_PORT', '3306')
database = os.getenv('DB_NAME', 'insight_places')

# Security Check: Validating if environment variables are present
if all([user, password, host, database]):
    print("🔒 Environment variables loaded successfully.")
    print("   System ready for database connection.")
else:
    print("❌ Error: One or more credentials are missing from the .env file.")


🔒 Environment variables loaded successfully.
   System ready for database connection.


### 1.3) Database Engine Initialization
Using the validated credentials, I will use SQLAlchemy to create the engine. This object will be the communication bridge between Python and MySQL, allowing for efficient data transfer.


In [5]:
# Creating the database connection engine using validated credentials
from urllib.parse import quote_plus

# Encodes the password to prevent URL syntax errors with special characters
safe_password = quote_plus(password)

engine = create_engine(f"mysql+pymysql://{user}:{safe_password}@{host}:{port}/{database}")
print("🚀 Engine connection is ready!")


🚀 Engine connection is ready!


## 2) Execution: Refining and Enriching the `Reviews` Table

This section executes the step-by-step transformation pipeline to modernize the database schema, model realistic guest evaluation behaviors, synchronize sentiment logic between ratings and text, and enforce temporal sequence rules.


### 2.1 Schema Modernization (DDL Adjustments)

In this step, I modify the `reviews` table structure by verifying and adding missing attributes (`booking_id`, `host_review`, `created_at`) to ensure strict relational linkage with completed bookings, bidirectional feedback mechanisms, and accurate evaluation timestamps.


In [7]:
def update_reviews_schema():
    with engine.begin() as conn:
        print("Checking existing columns in 'reviews' table...")
        
        # Inspection of current columns
        columns_query = text("""
            SELECT COLUMN_NAME 
            FROM INFORMATION_SCHEMA.COLUMNS 
            WHERE TABLE_SCHEMA = DATABASE() AND TABLE_NAME = 'reviews'
        """)
        existing_columns = [row[0] for row in conn.execute(columns_query).fetchall()]
        
        # Dictionary of required new columns and their SQL definitions
        columns_to_add = {
            'booking_id': 'VARCHAR(255)',
            'host_review': 'TEXT',
            'created_at': 'DATE'
        }
        
        # Add missing columns
        for col_name, col_type in columns_to_add.items():
            if col_name not in existing_columns:
                conn.execute(text(f"ALTER TABLE reviews ADD COLUMN {col_name} {col_type};"))
                print(f"  `{col_name}` column successfully added.")
            else:
                print(f"  `{col_name}` column already exists.")

        print("✅ Reviews schema modernization completed successfully!")

update_reviews_schema()


Checking existing columns in 'reviews' table...
  `booking_id` column successfully added.
  `host_review` column successfully added.
  `created_at` column successfully added.
✅ Reviews schema modernization completed successfully!


### 2.2 Synthetic Data Generation & Business Logic Enrichment

In this step, I generate synthetic customer evaluations based exclusively on completed bookings (`booking_status = 'Completed'`). The script models realistic guest behaviors (65% Rating & Text, 15% Rating Only, 5% Text Only, 15% Non-response), enforces strict sentiment alignment between ratings and reviews, captures host feedback, and sets review dates within a 1-to-7-day post-checkout window.


In [6]:
from datetime import datetime, timedelta

def populate_enriched_reviews():
    with engine.begin() as conn:
        print("📥 Fetching completed bookings for review generation...")
        
        # Fetching completed bookings
        completed_bookings = pd.read_sql("""
            SELECT booking_id, listing_id, customer_id, end_date 
            FROM bookings 
            WHERE booking_status = 'Completed'
        """, conn)
        
        if completed_bookings.empty:
            print("❌ Error: No 'Completed' bookings found. Please check 'bookings' table.")
            return

        print(f"📊 Found {len(completed_bookings)} completed bookings eligible for review.")

        # Sentiment mappings in English (Rating -> Review Text)
        sentiment_mapping = {
            1: ["Poor location; terrible infrastructure; very noisy place."],
            2: ["Too far from main tourist attractions."],
            3: ["Good location; slightly far from tourist spots; good value for money."],
            4: ["Good infrastructure; prime location near top tourist attractions."],
            5: ["Excellent infrastructure; privileged view; convenient airport shuttle service."]
        }

        # Host review options in English
        host_comments = [
            "Great guest, caused no damage;",
            "Terrible guest, caused damage and/or exceeded guest capacity."
        ]

        new_reviews = []
        review_counter = 1001

        print("⚡ Generating synthetic reviews based on behavioral distribution...")

        for _, row in completed_bookings.iterrows():
            # Behavioral probability distribution (A, B, C, D)
            # C: 65%, A: 15%, D: 15%, B: 5%
            behavior = random.choices(
                ['Option_C', 'Option_A', 'Option_D', 'Option_B'],
                weights=[0.65, 0.15, 0.15, 0.05],
                k=1
            )[0]

            # Option D: Guest did not leave a review
            if behavior == 'Option_D':
                continue

            # Review submission date (1 to 7 days after checkout)
            end_dt = datetime.strptime(str(row['end_date']), '%Y-%m-%d')
            review_dt = end_dt + timedelta(days=random.randint(1, 7))

            rating = None
            review_text = None

            if behavior == 'Option_C':
                # Rating & Review Text aligned
                rating = random.choices([1, 2, 3, 4, 5], weights=[0.05, 0.10, 0.20, 0.35, 0.30])[0]
                review_text = random.choice(sentiment_mapping[rating])

            elif behavior == 'Option_A':
                # Rating Only
                rating = random.choices([1, 2, 3, 4, 5], weights=[0.05, 0.10, 0.20, 0.35, 0.30])[0]
                review_text = None

            elif behavior == 'Option_B':
                # Text Only
                rating = None
                random_rating_key = random.choices([1, 2, 3, 4, 5], weights=[0.05, 0.10, 0.20, 0.35, 0.30])[0]
                review_text = random.choice(sentiment_mapping[random_rating_key])

            # Host review evaluation (50% chance of host leaving feedback)
            host_review = random.choices([None, host_comments[0], host_comments[1]], weights=[0.50, 0.45, 0.05])[0]

            new_reviews.append({
                'review_id': f"REV_{review_counter}",
                'booking_id': row['booking_id'],
                'listing_id': row['listing_id'],
                'customer_id': row['customer_id'],
                'rating': rating,
                'review_text': review_text,
                'host_review': host_review,
                'created_at': review_dt.strftime('%Y-%m-%d')
            })

            review_counter += 1

        # Convert to DataFrame
        enriched_reviews_df = pd.DataFrame(new_reviews)

        # Clear legacy reviews and insert enriched dataset
        print("🧹 Truncating legacy 'reviews' table...")
        conn.execute(text("DELETE FROM reviews;"))

        print(f"🚀 Inserting {len(enriched_reviews_df)} enriched review records into MySQL...")
        enriched_reviews_df.to_sql('reviews', con=conn, if_exists='append', index=False)

        print("✅ Reviews enrichment and populate pipeline executed successfully!")

populate_enriched_reviews()

📥 Fetching completed bookings for review generation...
📊 Found 43216 completed bookings eligible for review.
⚡ Generating synthetic reviews based on behavioral distribution...
🧹 Truncating legacy 'reviews' table...
🚀 Inserting 36681 enriched review records into MySQL...
✅ Reviews enrichment and populate pipeline executed successfully!


### 2.3 Data Quality Check & Structural Inspection

In this final step, I execute verification queries to confirm row count, inspect behavioral archetype distributions, validate sentiment consistency, and verify that review submission timestamps strictly occur post-checkout.


In [7]:
def validate_reviews_data():
    with engine.begin() as conn:
        print("🔍 Executing Data Quality Inspection on 'reviews' table...\n")
        
        # 1. Total row count check
        count_result = conn.execute(text("SELECT COUNT(*) FROM reviews;")).scalar()
        print(f"📊 Total Records in 'reviews': {count_result} rows.")
        
        # 2. Rating distribution breakdown
        rating_df = pd.read_sql("""
            SELECT 
                COALESCE(CAST(rating AS CHAR), 'No Rating (Text Only)') as rating_level, 
                COUNT(*) as total_count 
            FROM reviews 
            GROUP BY rating;
        """, conn)
        print("\n📈 Rating Distribution Breakdown:")
        print(rating_df.to_string(index=False))
        
        # 3. Temporal rule check (Verifying if created_at is strictly after booking end_date)
        temporal_check = conn.execute(text("""
            SELECT COUNT(*) 
            FROM reviews r 
            JOIN bookings b ON r.booking_id = b.booking_id 
            WHERE r.created_at < b.end_date;
        """)).scalar()
        
        if temporal_check == 0:
            print("\n✅ Temporal Rule Passed: 100% of reviews were created after booking checkout.")
        else:
            print(f"\n⚠️ Temporal Rule Failed: {temporal_check} records have timestamps prior to checkout.")

        # 4. Sample record preview
        print("\n👀 Preview of 5 Enriched Review Records:")
        sample_df = pd.read_sql("SELECT review_id, booking_id, rating, review_text, host_review, created_at FROM reviews LIMIT 5;", conn)
        print(sample_df.to_string(index=False))

validate_reviews_data()


🔍 Executing Data Quality Inspection on 'reviews' table...

📊 Total Records in 'reviews': 36681 rows.

📈 Rating Distribution Breakdown:
         rating_level  total_count
                    1         1765
                    5        10461
                    3         6713
                    4        12076
                    2         3445
No Rating (Text Only)         2221

✅ Temporal Rule Passed: 100% of reviews were created after booking checkout.

👀 Preview of 5 Enriched Review Records:
review_id booking_id  rating                                                                    review_text                    host_review created_at
REV_10000   BK_26279       1                      Poor location; terrible infrastructure; very noisy place.                           None 2025-11-16
REV_10001    BK_2628       5 Excellent infrastructure; privileged view; convenient airport shuttle service.                           None 2025-12-07
REV_10002   BK_26280       3          Good location